# Scénario 11f : `PAY_1` d'origine ou `PAY_1` corrigée, chaque modèle poussé au maximum

*Journal : [`tableau_resultats.md`](tableau_resultats.md). Boucle et règle de décision reprises d'[`essai_jeu_de_base.ipynb`](../comparatif_global/essai_jeu_de_base.ipynb).*

**Question** : pour prédire le défaut (`dpnm`), vaut-il mieux donner aux modèles la codification de septembre **d'origine** (`PAY_1`, avec ses codifications 1) ou la codification **corrigée** (`PAY_1_corrigee`) ?

**Deux modèles, une seule différence** (décisions de l'utilisateur) :
- **« PAY_1 d'origine »** : les 23 variables d'origine ;
- **« PAY_1 corrigée »** : les mêmes 23 variables, `PAY_1_corrigee` **à la place** de `PAY_1`.

Chaque modèle est **poussé au maximum de sa capacité** : la boucle d'optimisation d'`essai_jeu_de_base` (critère : précision moyenne ; surapprentissage limité à 7 % relatif ; grilles prolongées tant que le meilleur réglage est au bord), lancée **séparément pour chaque version**, avec des grilles de départ élargies (MLP jusqu'à deux couches de 256 et 128 neurones, RandomForest jusqu'à 500 arbres, CatBoost jusqu'à une profondeur de 8). Les deux versions sont ensuite comparées à la fin, pli par pli, chacune avec ses meilleurs réglages.

**Données, identiques dans les deux versions** : les 30 000 clients de `data/creditcard_pret_ingestion.csv` ; codifications 1 d'avril à août corrigées par le `10` (sans la règle fixe des retards), comme dans le `11e` ; `PAY_1_corrigee` lue dans `sorties/pay1_corrigee.csv`, écrite par le `11e` (codification d'août pour les 1 sains en août ; -1 si le MLP et CatBoost du `11c` annoncent une sortie du retard, 2 sinon).

**Encodages, identiques dans les deux versions** : **toutes les codifications en catégories** (une colonne par valeur : `PAY_1` ou `PAY_1_corrigee`, `PAY_2` à `PAY_6`) ; paiements et plafond au logarithme ; factures au logarithme signé ; âge mis à l'échelle ; `SEX`, `EDUCATION`, `MARRIAGE` en une colonne par catégorie.

**Mêmes 5 plis** (`random_state=42`) et même règle de décision qu'`essai_jeu_de_base` : au plus 1 bon client signalé pour 1 défaut, à la marge, seuil choisi dans des plis internes.

⚠️ Le test du ML est inclus dans les 30 000 clients, comme dans `essai_jeu_de_base` : cet essai éclaire la question, il ne choisit pas le modèle final.

## 1. Chargement du jeu de base et variables des trois variantes

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

original = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")   # niveau 0 du nettoyage (02_01_nettoyage.ipynb)
assert original["ID"].is_unique and not original.isna().any().any()
y_train = original["dpnm"].reset_index(drop=True)
print(f"Clients : {len(original)} ; colonnes : {original.shape[1]} (ID, 23 variables, dpnm)")
print(f"Défauts : {y_train.sum()} ({y_train.mean() * 100:.2f} %)")

Clients : 30000 ; colonnes : 25 (ID, 23 variables, dpnm)
Défauts : 6636 (22.12 %)


## 1 bis. Codifications 1 d'avril à août corrigées, `PAY_1_corrigee` lue dans le `11e`

In [2]:
PAY = [f"PAY_{n}" for n in range(1, 7)]
avant = original[PAY].copy()

def saine_anterieure(ligne, n):
    for k in range(n + 1, 7):
        if ligne[f"PAY_{k}"] <= 0:
            return ligne[f"PAY_{k}"]
    return None

# Correctifs du 10, des mois les plus anciens aux plus récents, sans la règle des retards du 09 (identiques au 11e)
for n in range(6, 1, -1):
    for k in original.index[(original[f"PAY_{n}"] == 1) & (original[f"BILL_AMT{n - 1}"] <= 0)]:
        v = saine_anterieure(original.loc[k], n)
        if v is not None:
            original.at[k, f"PAY_{n}"] = v
for n in range(5, 1, -1):
    regle = (original[f"PAY_{n}"] == 1) & (original[f"BILL_AMT{n}"] <= 0) & (original[f"PAY_{n + 1}"] <= 0)
    original.loc[regle, f"PAY_{n}"] = original.loc[regle, f"PAY_{n + 1}"]
DECISIONS = {6783: {4: 0, 3: 0, 2: 0}, 11498: {4: 2, 3: 2, 2: 2}, 892: {3: 0, 2: 0}, 1967: {3: 0, 2: 0},
             5645: {2: 0}, 19195: {2: 2}, 25176: {2: 2}}
for id_client, valeurs in DECISIONS.items():
    ligne = original.index[original["ID"] == id_client]
    assert len(ligne) == 1, id_client
    for n, v in valeurs.items():
        original.loc[ligne, f"PAY_{n}"] = v
assert int((original[PAY[1:]] == 1).sum().sum()) == 0 and (original["PAY_1"] == avant["PAY_1"]).all()

colonne = pd.read_csv(Path.cwd() / "sorties" / "pay1_corrigee.csv")
assert (colonne["ID"].to_numpy() == original["ID"].to_numpy()).all() and (colonne["PAY_1"].to_numpy() == original["PAY_1"].to_numpy()).all()
original["PAY_1_corrigee"] = colonne["PAY_1_corrigee"].to_numpy()
assert not (original["PAY_1_corrigee"] == 1).any()
print(f"Codifications d'avril à août corrigées : {int((original[PAY] != avant).sum().sum())} (clients : {int((original[PAY] != avant).any(axis=1).sum())})")
print(f"PAY_1_corrigee différente de PAY_1 : {int((original['PAY_1_corrigee'] != original['PAY_1']).sum())} clients (les PAY_1 = 1)")
display(pd.crosstab(original["PAY_1"].rename("PAY_1 d'origine"), original["PAY_1_corrigee"].rename("PAY_1_corrigee")))

Codifications d'avril à août corrigées : 34 (clients : 28)
PAY_1_corrigee différente de PAY_1 : 3688 clients (les PAY_1 = 1)


PAY_1_corrigee,-2,-1,0,2,3,4,5,6,7,8
PAY_1 d'origine,,,,,,,,,,
-2,2759,0,0,0,0,0,0,0,0,0
-1,0,5686,0,0,0,0,0,0,0,0
0,0,0,14737,0,0,0,0,0,0,0
1,1233,753,7,1695,0,0,0,0,0,0
2,0,0,0,2667,0,0,0,0,0,0
3,0,0,0,0,322,0,0,0,0,0
4,0,0,0,0,0,76,0,0,0,0
5,0,0,0,0,0,0,26,0,0,0
6,0,0,0,0,0,0,0,11,0,0


## 1 ter. Variables des deux versions

In [3]:
# ==============================================================================
# VARIABLES : les 23 variables d'origine, rangées selon leur encodage (celui des scénarios, méthode v8)
# ==============================================================================
PAY = [f"PAY_{n}" for n in range(1, 7)]

# PAY_n : nombres ordonnés (-2 à 8), mis à l'échelle (v6)
# PAY_AMTn : paiements très étirés, logarithme (v7)
# BILL_AMTn : factures, logarithme signé, car elles peuvent être négatives (v8)
# SEX, EDUCATION, MARRIAGE : catégories sans ordre, une colonne par catégorie (regroupées au niveau 0)
TYPES_ORIGINE = {
    "num": ["AGE"],
    "log": [f"PAY_AMT{n}" for n in range(1, 7)] + ["LIMIT_BAL"],   # 11f : plafond au logarithme, comme les paiements
    "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)],
    "nom": ["SEX", "EDUCATION", "MARRIAGE"] + PAY + ["PAY_1_corrigee"],   # 11f : toutes les codifications en catégories
}

# Les trois variantes : colonnes gardées, chacune avec son encodage d'origine
# Deux versions : PAY_1 d'origine, ou PAY_1_corrigee à sa place ; les 22 autres variables sont les mêmes
VARIANTES = {
    "PAY_1 d'origine": lambda c: c != "PAY_1_corrigee",
    "PAY_1 corrigée": lambda c: c != "PAY_1",
}
TYPES_VARIANTE = {v: {t: [c for c in cols if garder(c)] for t, cols in TYPES_ORIGINE.items()} for v, garder in VARIANTES.items()}
COLONNES_VARIANTE = {v: [c for cols in types.values() for c in cols] for v, types in TYPES_VARIANTE.items()}

# Contrôles : colonnes présentes, ni identifiant ni cible, les 23 variables couvertes par la variante complète
assert sorted(set(COLONNES_VARIANTE["PAY_1 d'origine"]) | set(COLONNES_VARIANTE["PAY_1 corrigée"])) == sorted(c for c in original.columns if c not in ("ID", "dpnm"))
assert len(COLONNES_VARIANTE["PAY_1 d'origine"]) == len(COLONNES_VARIANTE["PAY_1 corrigée"]) == 23
assert set(COLONNES_VARIANTE["PAY_1 d'origine"]) ^ set(COLONNES_VARIANTE["PAY_1 corrigée"]) == {"PAY_1", "PAY_1_corrigee"}
assert (original[TYPES_ORIGINE["log"]] >= 0).all().all(), "Le logarithme ne s'applique qu'à des montants positifs ou nuls"

display(pd.DataFrame({
    "Variante": list(VARIANTES),
    "Nombre de variables": [len(COLONNES_VARIANTE[v]) for v in VARIANTES],
    "Variables": [", ".join(COLONNES_VARIANTE[v]) for v in VARIANTES],
}).set_index("Variante"))

,Nombre de variables,Variables
Variante,,
PAY_1 d'origine,23,"AGE, PAY_AMT1, PAY_AMT2, PAY_AMT3, PAY_AMT4, P..."
PAY_1 corrigée,23,"AGE, PAY_AMT1, PAY_AMT2, PAY_AMT3, PAY_AMT4, P..."


## 2. Pipeline, critère et boucle : repris des scénarios

Le pipeline et la boucle sont ceux des scénarios (`ml_13`, méthode v8), à quatre différences près : le **critère de la boucle**, la précision moyenne (`CRITERE`, au lieu de la précision au rappel minimal) ; le **seuil de surapprentissage à 7 %**, mesuré sur ce critère ; les **4 modèles** suivis ; un pipeline construit **pour chaque variante** (seules ses colonnes y entrent). Le plancher contre le sous-apprentissage ne change pas de sens : la précision moyenne d'un tri au hasard vaut le taux de défaut. Aucune valeur n'est remplacée : le jeu de base n'a pas de vide.

In [4]:
from catboost import CatBoostClassifier
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", None)   # textes affichés en entier

# Pondération des défauts pour CatBoost (part des non-défauts / part des défauts)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

# Transformation de chaque type de colonne (un nouvel objet à chaque appel)
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}

def preprocesseur(variante):
    # Une branche par type de colonne, seulement pour les types présents dans la variante
    return ColumnTransformer(transformers=[(nom, TRANSFORMATIONS[nom](), colonnes)
                                           for nom, colonnes in TYPES_VARIANTE[variante].items() if colonnes])

# Critère optimisé par la boucle : la précision moyenne (aire sous la courbe précision-rappel), sur tous les rappels
CRITERE = "average_precision"
scorers = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
}

# Rappel minimal des scénarios (60 %) : sert seulement à la comparaison avec les scénarios (section 6)
RAPPEL_MINIMAL = 0.60

def seuil_pour_rappel(y, proba, rappel=RAPPEL_MINIMAL):
    # Seuil le plus haut qui détecte au moins la part voulue des défauts
    proba_defauts = np.sort(proba[np.asarray(y) == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modèles de l'essai : les meilleurs des scénarios, avec leurs réglages de la première itération
MODELES_A_TESTER = ["LogisticRegression", "MLPClassifier", "RandomForest", "CatBoost"]

def modeles_de_base(variante):
    prep = preprocesseur(variante)
    tous = {
        "LogisticRegression": make_pipeline(prep, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        "MLPClassifier": make_pipeline(prep, MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=500, random_state=42, early_stopping=True)),
        "RandomForest": make_pipeline(prep, RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
        "CatBoost": make_pipeline(prep, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1, allow_writing_files=False)),
    }
    return {nom: tous[nom] for nom in MODELES_A_TESTER}

# ------------------------------------------------------------------------------
# Alertes et tableaux de scores (ceux des scénarios)
SEUIL_SURAPPRENTISSAGE = 0.07   # écart relatif train - val du critère : 7 % pour cet essai (5 % dans les scénarios) ; le double : surapprentissage fort
SEUIL_INSTABILITE = 0.02        # écart-type du critère sur les plis au-delà duquel on alerte (alerte seule, pas une contrainte)
METRIQUES = {"average_precision": "Précision moyenne", "roc_auc": "ROC AUC"}
LIBELLE_SCORE = "Précision des défauts prédits"
TAUX_DEFAUT = y_train.mean()    # précision, et précision moyenne, d'un tri au hasard

def apprend(score_val, std_val):
    # Plancher contre le sous-apprentissage : le critère doit dépasser le hasard (taux de défaut) de plus d'un écart-type
    return score_val - TAUX_DEFAUT > std_val

def alerte(ecart):
    if ecart > 2 * SEUIL_SURAPPRENTISSAGE:
        return "⚠️⚠️ SURAPPRENTISSAGE FORT"
    if ecart > SEUIL_SURAPPRENTISSAGE:
        return "⚠️ SURAPPRENTISSAGE"
    return "✅ OK"

def ligne_scores(nom, moy_val, std_val, moy_train):
    # Une ligne de tableau : chaque métrique en entraînement et en validation, le critère, son écart-type et l'alerte
    ligne = {"Modèle": nom}
    for cle, libelle in METRIQUES.items():
        ligne[f"{libelle} (train)"] = moy_train[cle]
        ligne[f"{libelle} (val)"] = moy_val[cle]
    ligne["Critère (val)"], ligne["Critère (train)"] = moy_val[CRITERE], moy_train[CRITERE]
    ligne["Écart-type critère (val)"] = std_val
    ligne["Écart train - val"] = moy_train[CRITERE] - moy_val[CRITERE]
    ligne["Écart relatif"] = ligne["Écart train - val"] / moy_val[CRITERE]
    ligne["Apprend"] = apprend(moy_val[CRITERE], std_val)
    ligne["Alerte"] = alerte(ligne["Écart relatif"]) if ligne["Apprend"] else "⚠️ N'APPREND RIEN (niveau du hasard)"
    if std_val > SEUIL_INSTABILITE:
        ligne["Alerte"] += " ; ⚠️ INSTABLE"
    return ligne

def pct(serie):
    return (serie * 100).round(1)

def parametres(params):
    # Paramètres sur une ligne : « depth 4, 5, 6 ; l2_leaf_reg 15 »
    return " ; ".join(f"{p} {', '.join(map(str, v)) if isinstance(v, list) else v}" for p, v in params.items())

def montrer(df, titre):
    display(Markdown(f"**{titre}**"))
    display(df.set_index(df.columns[0]))

In [5]:
from itertools import product
from sklearn.model_selection import GridSearchCV

TOURS_SECURITE = 15   # garde-fou contre une boucle sans fin : la boucle s'arrête d'elle-même avant ; s'il est atteint, revoir les grilles ou les bornes

# Grilles du premier tour : celles des scénarios, élargies pour pousser chaque modèle au maximum (11f)
grilles = {
    "LogisticRegression": {"logisticregression__C": [0.001, 0.01, 0.1, 1.0, 10.0]},
    "MLPClassifier": {"mlpclassifier__hidden_layer_sizes": [(32,), (32, 16), (64, 32), (128, 64), (256, 128)], "mlpclassifier__alpha": [0.001, 0.01, 0.1, 1.0]},
    "RandomForest": {"randomforestclassifier__max_depth": [6, 8, 10, 12], "randomforestclassifier__min_samples_leaf": [10, 20, 50, 100], "randomforestclassifier__n_estimators": [300, 500]},
    "CatBoost": {"catboostclassifier__depth": [4, 6, 8], "catboostclassifier__l2_leaf_reg": [5, 10, 15, 30], "catboostclassifier__learning_rate": [0.02, 0.03, 0.05],
                 "catboostclassifier__iterations": [1000]},   # 1 000 arbres : valeur par défaut, réduite par la boucle en cas de surapprentissage
}

# Comment faire évoluer chaque paramètre numérique d'un tour à l'autre (ceux des scénarios) :
# - simplifier : sens qui réduit le surapprentissage (+1 = augmenter, -1 = diminuer) ;
# - pas : écart entre deux valeurs (« x » = multiplier, « + » = ajouter) ;
# - bornes : valeurs minimale et maximale autorisées.
# Les paramètres absents de ce tableau (architecture du MLP, nombre d'arbres du RandomForest) sont fixés à leur meilleure valeur après le premier tour.
REGLAGES_PARAMETRES = {
    "logisticregression__C":                    {"simplifier": -1, "pas": ("x", 3),   "bornes": (1e-4, 1e4),  "entier": False},
    "mlpclassifier__alpha":                     {"simplifier": +1, "pas": ("x", 3),   "bornes": (1e-5, 100),  "entier": False},
    "randomforestclassifier__max_depth":        {"simplifier": -1, "pas": ("+", 2),   "bornes": (2, 30),      "entier": True},
    "randomforestclassifier__min_samples_leaf": {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 2000),    "entier": True},
    "catboostclassifier__depth":                {"simplifier": -1, "pas": ("+", 1),   "bornes": (2, 10),      "entier": True},
    "catboostclassifier__l2_leaf_reg":          {"simplifier": +1, "pas": ("x", 1.5), "bornes": (1, 200),     "entier": False},
    "catboostclassifier__learning_rate":        {"simplifier": -1, "pas": ("x", 1.5), "bornes": (0.003, 0.3), "entier": False},
    "catboostclassifier__iterations":           {"simplifier": -1, "pas": ("x", 1.5), "bornes": (100, 3000),  "entier": True},
}

def nom_court(parametre):
    return parametre.split("__")[-1]

def arrondir(valeur, entier):
    return int(round(valeur)) if entier else float(f"{valeur:.3g}")

def voisin(valeur, reglage, sens):
    # Valeur suivante dans un sens (+1 = plus grande, -1 = plus petite), dans les bornes
    mode, pas = reglage["pas"]
    if mode == "x":
        nouvelle = valeur * pas if sens > 0 else valeur / pas
    else:
        nouvelle = valeur + pas if sens > 0 else valeur - pas
    bas, haut = reglage["bornes"]
    nouvelle = arrondir(min(max(nouvelle, bas), haut), reglage["entier"])
    return None if nouvelle == valeur else nouvelle

def choisir_index(r):
    # Choix sous contrainte, parmi les réglages qui battent le hasard : le meilleur critère sans surapprentissage ;
    # sinon le moins surappris ; si aucun ne bat le hasard, le meilleur score
    val, std = r[f"mean_test_{CRITERE}"], r[f"std_test_{CRITERE}"]
    ecarts = (r[f"mean_train_{CRITERE}"] - val) / val   # écarts relatifs
    utiles = apprend(val, std)
    sous_seuil = np.where(utiles & (ecarts <= SEUIL_SURAPPRENTISSAGE))[0]
    if len(sous_seuil):
        return int(sous_seuil[np.argmax(val[sous_seuil])])
    if utiles.any():
        idx = np.where(utiles)[0]
        return int(idx[np.argmin(ecarts[idx])])
    return int(np.argmax(val))

def bords_de_grille(grille, meilleurs):
    # Paramètres numériques dont la meilleure valeur est la plus petite ou la plus grande de la grille (+1 = maximum, -1 = minimum)
    bords = {}
    for p, valeurs in grille.items():
        if p in REGLAGES_PARAMETRES and len(valeurs) > 1:
            if meilleurs[p] == max(valeurs):
                bords[p] = +1
            elif meilleurs[p] == min(valeurs):
                bords[p] = -1
    return bords

def grille_suivante(grille, meilleurs, bords, surapprend):
    # Grille du tour suivant et explication des corrections ; None si plus rien ne peut bouger
    nouvelle, actions = {}, []
    for p, valeurs in grille.items():
        v = meilleurs[p]
        reglage = REGLAGES_PARAMETRES.get(p)
        if reglage is None:
            nouvelle[p] = [v]                            # catégorie ou paramètre sans effet : fixé
            continue
        sens = None
        if surapprend and reglage["simplifier"] != 0:
            sens = reglage["simplifier"]                 # pousser vers un modèle plus simple
            raison = "surapprentissage"
        elif p in bords:
            sens = bords[p]                              # prolonger la grille du côté du bord
            raison = "bord de grille"
        if sens is None:
            nouvelle[p] = [v]
            continue
        v1 = voisin(v, reglage, sens)
        v2 = voisin(v1, reglage, sens) if v1 is not None else None
        valeurs_tour = [x for x in (v, v1, v2) if x is not None]
        nouvelle[p] = sorted(set(valeurs_tour))
        if len(nouvelle[p]) > 1:
            actions.append(f"{nom_court(p)} {v} → {nouvelle[p]} ({raison})")
    if not actions:
        return None, ["bornes atteintes : la grille ne peut plus bouger"]
    return nouvelle, actions

def lancer_grille(nom, modele, grille, X):
    grid = GridSearchCV(modele, grille, cv=cv, scoring=scorers, refit=choisir_index, return_train_score=True, n_jobs=-1)
    grid.fit(X, y_train)
    r, i = grid.cv_results_, grid.best_index_
    ligne = ligne_scores(nom,
                         {k: r[f"mean_test_{k}"][i] for k in METRIQUES},
                         r[f"std_test_{CRITERE}"][i],
                         {k: r[f"mean_train_{k}"][i] for k in METRIQUES})
    ligne["Meilleurs paramètres"] = {nom_court(k): v for k, v in grid.best_params_.items()}
    return grid, ligne

def cle_reglage(params):
    # Identifiant d'une combinaison de paramètres, pour savoir si elle a déjà été testée
    return str(sorted(params.items()))

def reglages_nouveaux(grille, deja_testes):
    # Combinaisons de la grille qui n'ont encore été testées à aucun tour
    combinaisons = [dict(zip(grille, valeurs)) for valeurs in product(*grille.values())]
    return [c for c in combinaisons if cle_reglage(c) not in deja_testes]

def choisir_tour(tours):
    # Version retenue parmi les tours joués : d'abord ceux qui battent le hasard et sans surapprentissage
    # (sinon ceux qui battent le hasard, sinon tous) ; parmi eux, ceux à moins d'un écart-type du meilleur score
    # (à égalité avec lui) ; et parmi ces derniers, celui qui surapprend le moins
    for garder in (lambda l: l["Apprend"] and l["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE, lambda l: l["Apprend"], lambda l: True):
        groupe = [t for t in tours if garder(t[1])]
        if groupe:
            break
    meilleur = max(groupe, key=lambda t: t[1]["Critère (val)"])[1]
    egaux = [t for t in groupe if meilleur["Critère (val)"] - t[1]["Critère (val)"] <= meilleur["Écart-type critère (val)"]]
    return min(egaux, key=lambda t: t[1]["Écart relatif"])

def boucle(variante):
    # Boucle des scénarios, pour un jeu de variables : version retenue de chaque modèle, lignes de scores et historique
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    best_models, lignes_retenues, historique = {}, [], []
    for nom, modele in modeles_de_base(variante).items():
        grille, retenu, fin, deja_testes, reglage_precedent, tours = grilles[nom], None, None, set(), None, []
        for tour in range(1, TOURS_SECURITE + 1):
            grid, ligne = lancer_grille(nom, modele, grille, X)
            ligne["Tour"] = tour
            deja_testes |= {cle_reglage(c) for c in grid.cv_results_["params"]}
            surapprend = ligne["Écart relatif"] > SEUIL_SURAPPRENTISSAGE
            bords = bords_de_grille(grille, grid.best_params_)
            precedent = retenu
            tours.append((grid, ligne))
            retenu = choisir_tour(tours)
            grille_tour = {nom_court(p): v for p, v in grille.items()}
            reglage = ligne["Meilleurs paramètres"]
            # Stagnation : seulement entre deux tours sans surapprentissage
            stagne = (precedent is not None and not surapprend
                      and precedent[1]["Écart relatif"] <= SEUIL_SURAPPRENTISSAGE
                      and ligne["Critère (val)"] - precedent[1]["Critère (val)"] <= precedent[1]["Écart-type critère (val)"])
            if not ligne["Apprend"]:
                fin = "sous-apprentissage : le réglage choisi ne bat plus le hasard"
            elif not surapprend and not bords:
                fin = "optimisé : ni surapprentissage ni paramètre au bord"
            elif reglage == reglage_precedent:
                fin = "convergé : le réglage choisi ne change plus"
            elif stagne:
                fin = "stagnation : gain plus petit que l'écart-type, entre deux tours sans surapprentissage"
            elif tour == TOURS_SECURITE:
                fin = "⚠️ garde-fou atteint : grilles ou bornes à revoir"
            else:
                grille, actions = grille_suivante(grille, grid.best_params_, bords, surapprend)
                if grille is None:
                    fin = actions[0]
                elif not reglages_nouveaux(grille, deja_testes):
                    fin = "convergé : la grille suivante ne contient aucun réglage nouveau"
            reglage_precedent = reglage
            historique.append({
                "Modèle": nom, "Tour": tour, "Grille testée": grille_tour, "Réglage choisi": reglage,
                "Critère (val)": ligne["Critère (val)"], "Écart-type": ligne["Écart-type critère (val)"],
                "Écart relatif": ligne["Écart relatif"], "Alerte": ligne["Alerte"],
                "Suite": fin if fin else "; ".join(actions),
            })
            if fin:
                break
        best_models[nom] = retenu[0].best_estimator_
        lignes_retenues.append(dict(retenu[1], **{"Arrêt": fin, "Tours joués": tour}))
        print(f"  {nom} : {tour} tour(s), {fin} ; version retenue : tour {retenu[1]['Tour']}")
    return best_models, pd.DataFrame(lignes_retenues), pd.DataFrame(historique)

## 3. Boucle d'optimisation, variante par variante

Pour chaque variante, la boucle règle les 4 modèles l'un après l'autre. Une ligne s'affiche pendant le calcul pour chaque modèle (nombre de tours, raison de l'arrêt, tour retenu). Puis, pour chaque variante :
- **Version retenue de chaque modèle** : la précision moyenne en validation (le critère), son écart-type, l'entraînement face à la validation, l'écart relatif (au-delà de 7 % : ⚠️ ; de 14 % : ⚠️⚠️), le ROC AUC en contrôle, et le réglage retenu ;
- **Historique de la boucle** : une ligne par tour et par modèle, avec la grille testée, le réglage choisi et la correction qui suit (ou la raison de l'arrêt).

Le calcul est long : la boucle tourne deux fois, sur 30 000 clients.

In [6]:
import time

resultats = {}
for variante in VARIANTES:
    debut = time.time()
    print(f"Variante « {variante} » ({len(COLONNES_VARIANTE[variante])} variables)")
    resultats[variante] = boucle(variante)
    print(f"  durée : {(time.time() - debut) / 60:.1f} min")

for variante, (_, df_retenus, df_historique) in resultats.items():
    display(Markdown(f"### Variante « {variante} »"))
    df = df_retenus.sort_values("Critère (val)", ascending=False)
    montrer(pd.DataFrame({
        "Modèle": df["Modèle"],
        "Précision moyenne val (%)": pct(df["Critère (val)"]),
        "Écart-type (pts)": pct(df["Écart-type critère (val)"]),
        "Précision moyenne train (%)": pct(df["Critère (train)"]),
        "Écart relatif (%)": pct(df["Écart relatif"]),
        "ROC AUC val": df["ROC AUC (val)"].round(3),
        "ROC AUC train": df["ROC AUC (train)"].round(3),
        "Alerte": df["Alerte"],
        "Tours joués": df["Tours joués"],
        "Tour retenu": df["Tour"],
        "Raison de l'arrêt": df["Arrêt"],
        "Réglage retenu": df["Meilleurs paramètres"].map(parametres),
    }), "Version retenue de chaque modèle (validation)")
    montrer(pd.DataFrame({
        "Modèle": df_historique["Modèle"],
        "Tour": df_historique["Tour"],
        "Grille testée": df_historique["Grille testée"].map(parametres),
        "Réglage choisi": df_historique["Réglage choisi"].map(parametres),
        "Précision moyenne val (%)": pct(df_historique["Critère (val)"]),
        "Écart relatif (%)": pct(df_historique["Écart relatif"]),
        "Alerte": df_historique["Alerte"],
        "Suite": df_historique["Suite"],
    }), "Historique de la boucle")

Variante « PAY_1 d'origine » (23 variables)
  LogisticRegression : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  MLPClassifier : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  RandomForest : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  CatBoost : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  durée : 7.9 min
Variante « PAY_1 corrigée » (23 variables)
  LogisticRegression : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  MLPClassifier : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  RandomForest : 1 tour(s), optimisé : ni surapprentissage ni paramètre au bord ; version retenue : tour 1
  CatBoost : 2 tour(s), convergé : le réglage choisi ne change plus ; version retenue : tour 1
  durée : 7.9 min


### Variante « PAY_1 d'origine »

**Version retenue de chaque modèle (validation)**

,Précision moyenne val (%),Écart-type (pts),Précision moyenne train (%),Écart relatif (%),ROC AUC val,ROC AUC train,Alerte,Tours joués,Tour retenu,Raison de l'arrêt,Réglage retenu
Modèle,,,,,,,,,,,
CatBoost,55.3,0.9,59.0,6.6,0.784,0.815,✅ OK,2,1,convergé : le réglage choisi ne change plus,depth 4 ; iterations 1000 ; l2_leaf_reg 30 ; learning_rate 0.02
MLPClassifier,55.2,1.0,58.5,6.0,0.777,0.797,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,"alpha 0.1 ; hidden_layer_sizes (256, 128)"
RandomForest,54.8,1.0,57.9,5.6,0.779,0.799,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,max_depth 8 ; min_samples_leaf 50 ; n_estimators 500
LogisticRegression,54.6,0.7,55.3,1.2,0.775,0.778,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,C 1.0


**Historique de la boucle**

...

### Variante « PAY_1 corrigée »

**Version retenue de chaque modèle (validation)**

,Précision moyenne val (%),Écart-type (pts),Précision moyenne train (%),Écart relatif (%),ROC AUC val,ROC AUC train,Alerte,Tours joués,Tour retenu,Raison de l'arrêt,Réglage retenu
Modèle,,,,,,,,,,,
CatBoost,53.9,0.6,57.7,7.0,0.782,0.812,✅ OK,2,1,convergé : le réglage choisi ne change plus,depth 4 ; iterations 1000 ; l2_leaf_reg 30 ; learning_rate 0.02
MLPClassifier,53.5,0.8,57.2,6.8,0.777,0.794,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,"alpha 0.01 ; hidden_layer_sizes (32, 16)"
RandomForest,52.9,1.2,56.5,6.8,0.777,0.797,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,max_depth 8 ; min_samples_leaf 50 ; n_estimators 300
LogisticRegression,52.4,0.7,53.2,1.5,0.770,0.774,✅ OK,1,1,optimisé : ni surapprentissage ni paramètre au bord,C 1.0


**Historique de la boucle**

,Tour,Grille testée,Réglage choisi,Précision moyenne val (%),Écart relatif (%),Alerte,Suite
Modèle,,,,,,,
LogisticRegression,1,"C 0.001, 0.01, 0.1, 1.0, 10.0",C 1.0,52.4,1.5,✅ OK,optimisé : ni surapprentissage ni paramètre au bord
MLPClassifier,1,"hidden_layer_sizes (32,), (32, 16), (64, 32), (128, 64), (256, 128) ; alpha 0.001, 0.01, 0.1, 1.0","alpha 0.01 ; hidden_layer_sizes (32, 16)",53.5,6.8,✅ OK,optimisé : ni surapprentissage ni paramètre au bord
RandomForest,1,"max_depth 6, 8, 10, 12 ; min_samples_leaf 10, 20, 50, 100 ; n_estimators 300, 500",max_depth 8 ; min_samples_leaf 50 ; n_estimators 300,52.9,6.8,✅ OK,optimisé : ni surapprentissage ni paramètre au bord
CatBoost,1,"depth 4, 6, 8 ; l2_leaf_reg 5, 10, 15, 30 ; learning_rate 0.02, 0.03, 0.05 ; iterations 1000",depth 4 ; iterations 1000 ; l2_leaf_reg 30 ; learning_rate 0.02,53.9,7.0,✅ OK,"depth 4 → [2, 3, 4] (bord de grille); l2_leaf_reg 30 → [30, 45.0, 67.5] (bord de grille); learning_rate 0.02 → [0.00887, 0.0133, 0.02] (bord de grille)"
CatBoost,2,"depth 2, 3, 4 ; l2_leaf_reg 30, 45.0, 67.5 ; learning_rate 0.00887, 0.0133, 0.02 ; iterations 1000",depth 4 ; iterations 1000 ; l2_leaf_reg 30 ; learning_rate 0.02,53.9,7.0,✅ OK,convergé : le réglage choisi ne change plus


**Lecture** : une section par variante. Le taux de défaut du jeu de base est la précision moyenne d'un tri au hasard : une précision moyenne proche de lui veut dire que le modèle ne trouve presque rien. Un modèle qui reste en surapprentissage malgré tous ses tours est signalé (⚠️) : sa grille de départ ou ses bornes sont peut-être à revoir pour ces données.

## 4. Seuil de décision optimisé, pli par pli

Repris d'`essai_jeu_de_base` : sur chaque pli, la version retenue de chaque modèle est réentraînée sur les 4 autres plis, avec un seuil choisi dans des plis internes (au plus 1 bon client signalé pour 1 défaut, à la marge), puis jugée sur le pli de validation. Le tableau donne, pour chaque version et chaque modèle, le seuil, le rappel, la précision des défauts prédits, la précision marginale, la précision moyenne et le ROC AUC.

In [7]:
from sklearn.base import clone
from sklearn.metrics import average_precision_score, fbeta_score, make_scorer, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import TunedThresholdClassifierCV

y = y_train.to_numpy()
plis = list(cv.split(original, y_train))   # les mêmes 5 plis que la boucle
cv_interne = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # plis internes, pour le choix du seuil

# Règle fixée à l'avance par le métier : au plus 1 bon client signalé pour 1 défaut attrapé, à la marge
BONS_CLIENTS_PAR_DEFAUT = 1
PRECISION_MARGINALE_VISEE = 1 / (1 + BONS_CLIENTS_PAR_DEFAUT)   # 50 % : précision des derniers clients ajoutés au seuil retenu
PART_MARGE = 0.10   # contrôle : précision mesurée sur les 10 % de clients signalés les plus proches du seuil

def gain_net(y_vrai, y_pred):
    # Défauts détectés moins les bons clients signalés (pondérés), rapporté au nombre de clients :
    # maximal au seuil où les clients ajoutés ne sont plus en défaut qu'avec la précision marginale visée
    y_vrai, y_pred = np.asarray(y_vrai), np.asarray(y_pred)
    detectes = (y_pred & (y_vrai == 1)).sum()
    fausses_alertes = (y_pred & (y_vrai == 0)).sum()
    return (detectes - BONS_CLIENTS_PAR_DEFAUT * fausses_alertes) / len(y_vrai)

SCORE_SEUIL = make_scorer(gain_net)

def precision_marginale(y_vrai, proba, signales):
    # Part de défauts parmi les PART_MARGE des clients signalés qui ont les probabilités les plus basses (les plus proches du seuil)
    idx = np.where(signales)[0]
    if len(idx) == 0:
        return np.nan
    derniers = idx[np.argsort(proba[idx])[:max(1, int(np.ceil(PART_MARGE * len(idx))))]]
    return y_vrai[derniers].mean()

def f2_au_rappel_minimal(y_vrai, proba):
    # Mesure des scénarios (section 6) : F2 au seuil qui détecte au moins le rappel minimal
    return fbeta_score(y_vrai, proba >= seuil_pour_rappel(y_vrai, proba), beta=2)

probas, mesures, modeles_plis = {}, {}, {}
for variante, (best_models, _, _) in resultats.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    for nom, modele in best_models.items():
        p, entraines = np.zeros(len(y)), []
        m_plis = {"Seuil": [], "Rappel": [], "Précision": [], "Précision marginale": [], "Signalés": [], "F1": [], "Précision moyenne": [], "ROC AUC": []}
        for entrainement, validation in plis:
            seuil_auto = TunedThresholdClassifierCV(clone(modele), scoring=SCORE_SEUIL, cv=cv_interne, n_jobs=-1)
            seuil_auto.fit(X.iloc[entrainement], y[entrainement])
            p[validation] = seuil_auto.predict_proba(X.iloc[validation])[:, 1]
            decision = seuil_auto.predict(X.iloc[validation]).astype(bool)
            y_val = y[validation]
            m_plis["Seuil"].append(seuil_auto.best_threshold_)
            m_plis["Rappel"].append(recall_score(y_val, decision))
            m_plis["Précision"].append(precision_score(y_val, decision, zero_division=0))
            m_plis["Précision marginale"].append(precision_marginale(y_val, p[validation], decision))
            m_plis["Signalés"].append(decision.mean())
            m_plis["F1"].append(fbeta_score(y_val, decision, beta=1))
            m_plis["Précision moyenne"].append(average_precision_score(y_val, p[validation]))
            m_plis["ROC AUC"].append(roc_auc_score(y_val, p[validation]))
            entraines.append(seuil_auto)
        probas[(variante, nom)], mesures[(variante, nom)], modeles_plis[(variante, nom)] = p, m_plis, entraines
        print(f"{variante}, {nom} : seuil {np.mean(m_plis['Seuil']):.3f}, rappel {np.mean(m_plis['Rappel']) * 100:.1f} %, "
              f"précision {np.mean(m_plis['Précision']) * 100:.1f} %, précision marginale {np.nanmean(m_plis['Précision marginale']) * 100:.1f} %")

def moy_et(valeurs, en_pct=False):
    facteur = 100 if en_pct else 1
    return f"{np.mean(valeurs) * facteur:.{1 if en_pct else 3}f} ± {np.std(valeurs) * facteur:.{1 if en_pct else 3}f}"

lignes = []
for (variante, nom), m in mesures.items():
    lignes.append({
        "Variante": variante, "Modèle": nom,
        "Seuil (moyenne)": round(np.mean(m["Seuil"]), 3),
        "Seuil (min - max)": f"{min(m['Seuil']):.3f} - {max(m['Seuil']):.3f}",
        "Rappel (%)": moy_et(m["Rappel"], True),
        f"{LIBELLE_SCORE} (%)": moy_et(m["Précision"], True),
        f"Précision marginale, {PART_MARGE:.0%} les plus proches du seuil (%)": moy_et(m["Précision marginale"], True),
        "Clients signalés (%)": moy_et(m["Signalés"], True),
        "F1": moy_et(m["F1"]),
        "Précision moyenne": moy_et(m["Précision moyenne"]),
        "ROC AUC": moy_et(m["ROC AUC"]),
    })
display(Markdown(f"**Détection au seuil optimisé (au plus {BONS_CLIENTS_PAR_DEFAUT} bon(s) client(s) pour 1 défaut à la marge, "
                 f"précision marginale visée {PRECISION_MARGINALE_VISEE:.0%}), en validation** (moyenne ± écart-type sur les 5 plis ; taux de défaut : {TAUX_DEFAUT * 100:.1f} %)"))
display(pd.DataFrame(lignes).set_index(["Variante", "Modèle"]))

PAY_1 d'origine, LogisticRegression : seuil 0.738, rappel 39.0 %, précision 65.7 %, précision marginale 48.9 %
PAY_1 d'origine, MLPClassifier : seuil 0.506, rappel 37.4 %, précision 65.9 %, précision marginale 47.3 %
PAY_1 d'origine, RandomForest : seuil 0.706, rappel 38.7 %, précision 65.1 %, précision marginale 52.7 %
PAY_1 d'origine, CatBoost : seuil 0.750, rappel 38.8 %, précision 65.3 %, précision marginale 51.0 %
PAY_1 corrigée, LogisticRegression : seuil 0.714, rappel 42.3 %, précision 61.1 %, précision marginale 51.8 %
PAY_1 corrigée, MLPClassifier : seuil 0.515, rappel 35.6 %, précision 64.2 %, précision marginale 48.6 %
PAY_1 corrigée, RandomForest : seuil 0.683, rappel 41.9 %, précision 60.1 %, précision marginale 48.3 %
PAY_1 corrigée, CatBoost : seuil 0.774, rappel 37.9 %, précision 63.3 %, précision marginale 52.8 %


**Détection au seuil optimisé (au plus 1 bon(s) client(s) pour 1 défaut à la marge, précision marginale visée 50%), en validation** (moyenne ± écart-type sur les 5 plis ; taux de défaut : 22.1 %)

Seuil (moyenne) Seuil (min - max)  Rappel (%) Précision des défauts prédits (%) Précision marginale, 10% les plus proches du seuil (%) Clients signalés (%)             F1 Précision moyenne        ROC AUC
Variante        Modèle                                                                                                                                                                                                                         
PAY_1 d'origine LogisticRegression            0.738     0.715 - 0.756  39.0 ± 2.0                        65.7 ± 1.2                                             48.9 ± 8.0           13.1 ± 0.7  0.489 ± 0.016     0.546 ± 0.007  0.775 ± 0.005
                MLPClassifier                 0.506     0.463 - 0.532  37.4 ± 2.7                        65.9 ± 2.4                                             47.3 ± 5.8           12.6 ± 1.3  0.476 ± 0.019     0.552 ± 0.010  0.777 ± 0.007
                RandomForest                  0.706     0.693 - 0.714  38.7 ± 1.5                        65.1 ± 1.9                                             52.7 ± 5.3           13.1 ± 0.7  0.485 ± 0.012     0.548 ± 0.010  0.779 ± 0.006
                CatBoost                      0.750     0.718 - 0.766  38.8 ± 1.9                        65.3 ± 0.9                                             51.0 ± 9.0           13.1 ± 0.8  0.486 ± 0.013     0.553 ± 0.009  0.784 ± 0.007
PAY_1 corrigée  LogisticRegression            0.714     0.703 - 0.733  42.3 ± 1.2                        61.1 ± 1.0                                             51.8 ± 6.3           15.3 ± 0.6  0.500 ± 0.007     0.524 ± 0.007  0.770 ± 0.006
                MLPClassifier                 0.515     0.493 - 0.549  35.6 ± 3.8                        64.2 ± 3.6                                             48.6 ± 7.5           12.4 ± 1.8  0.456 ± 0.027     0.535 ± 0.008  0.777 ± 0.006
                RandomForest                  0.683     0.626 - 0.740  41.9 ± 3.3                        60.1 ± 2.1                                             48.3 ± 5.1           15.5 ± 1.7  0.492 ± 0.017     0.529 ± 0.012  0.777 ± 0.006
                CatBoost                      0.774     0.767 - 0.783  37.9 ± 0.9                        63.3 ± 1.7                                             52.8 ± 5.0           13.3 ± 0.4  0.474 ± 0.010     0.539 ± 0.006  0.782 ± 0.007

**Comment lire** : comme dans `essai_jeu_de_base`. La précision marginale doit tourner autour de 50 % ; la comparaison des deux versions est en section 6.

In [8]:
# Enregistrement, client par client, des probabilités hors pli et des décisions au seuil optimisé (règle marginale),
# pour chaque version et chaque modèle, dans le dossier local sorties/. Aucun réentraînement :
# les décisions sont celles des modèles de chaque pli, gardés en mémoire à la section 4
NOM_ESSAI = "pay1_origine_ou_corrigee"
dossier_sortie = Path.cwd() / "sorties"
dossier_sortie.mkdir(parents=True, exist_ok=True)
sortie = pd.DataFrame({"ID": original["ID"].to_numpy(), "dpnm": y})
for (variante, nom), p in probas.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    decision = np.zeros(len(y), dtype=int)
    for m, (_, validation) in zip(modeles_plis[(variante, nom)], plis):
        decision[validation] = m.predict(X.iloc[validation])
    sortie[f"proba | {variante} | {nom}"] = p
    sortie[f"decision | {variante} | {nom}"] = decision
sortie.to_csv(dossier_sortie / f"{NOM_ESSAI}.csv", index=False)
print(f"Enregistré : {dossier_sortie / f'{NOM_ESSAI}.csv'} ({len(sortie)} clients, {len(probas)} couples variante × modèle)")

Enregistré : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\lab_ML\correction_codification_1\sorties\pay1_origine_ou_corrigee.csv (30000 clients, 8 couples variante × modèle)


## 5. Ce qui porte les prédictions : importance par permutation

**Principe** : sur chaque pli de validation, on reprend le modèle entraîné sur les 4 autres plis (section 4) et on **mélange** les valeurs d'une famille de colonnes entre les clients (les autres colonnes restent intactes). Le modèle perd ce que cette famille lui apportait : la **perte de précision moyenne** mesure son importance. La précision moyenne ne dépend d'aucun seuil : la mesure juge la capacité de détection, pas un point de fonctionnement, et elle est plus stable qu'un score pris en un seul point. Le mélange est répété 5 fois (graine fixée), puis la perte est moyennée sur les répétitions et sur les 5 plis. Une famille se mélange d'un bloc (les six mois ensemble, avec le même tirage pour chaque colonne), pour garder sa cohérence d'un mois à l'autre.

**Familles** : codification de septembre (`PAY_1` ou `PAY_1_corrigee`, selon la version), codifications d'avril à août (`PAY_2` à `PAY_6`), factures (`BILL_AMT1` à `6`), paiements (`PAY_AMT1` à `6`), plafond (`LIMIT_BAL`), démographie (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`). Le comportement du client, au sens des montants, ce sont les factures, les paiements et le plafond ; les codifications sont l'étiquette posée par la banque.

**Deux tableaux** :
- **par famille**, pour chaque variante et chaque modèle : perte de précision moyenne et son écart-type sur les plis, et la part de chaque famille dans la perte totale de la colonne ;
- **par colonne**, mélangée seule : le détail, pour voir quels mois et quels montants comptent.

**Ce que la permutation mesure** : ce que le modèle perd **quand la famille manque, les autres étant là**. Deux familles qui disent la même chose (les codifications et les paiements, par exemple) se remplacent l'une l'autre : chacune paraît alors peu importante, alors qu'ensemble elles portent beaucoup. C'est pourquoi cette section se lit avec la section 4 : les variantes « `PAY_n` seuls » et « Sans les `PAY_n` » disent ce que chaque bloc porte **seul**, la permutation dit ce qu'il apporte **en plus des autres**.

In [9]:
FAMILLES = {
    "Codification de septembre (PAY_1 ou PAY_1_corrigee)": ["PAY_1", "PAY_1_corrigee"],
    "Codifications d'avril à août (PAY_2 à PAY_6)": PAY[1:],
    "Factures (BILL_AMTn)": [f"BILL_AMT{n}" for n in range(1, 7)],
    "Paiements (PAY_AMTn)": [f"PAY_AMT{n}" for n in range(1, 7)],
    "Plafond (LIMIT_BAL)": ["LIMIT_BAL"],
    "Démographie": ["SEX", "EDUCATION", "MARRIAGE", "AGE"],
}
REPETITIONS = 5
generateur = np.random.default_rng(42)

def perte_par_permutation(modeles, X, colonnes):
    # Perte de précision moyenne sur chaque pli de validation quand les colonnes sont mélangées d'un bloc (même tirage pour toutes),
    # moyennée sur les répétitions
    pertes = []
    for m, (_, validation) in zip(modeles, plis):
        X_val, y_val = X.iloc[validation], y[validation]
        reference = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        perte = []
        for _ in range(REPETITIONS):
            melange = X_val.copy()
            ordre_melange = generateur.permutation(len(melange))
            melange[colonnes] = melange[colonnes].to_numpy()[ordre_melange]
            perte.append(reference - average_precision_score(y_val, m.predict_proba(melange)[:, 1]))
        pertes.append(np.mean(perte))
    return pertes

importance_familles, importance_colonnes = {}, {}
for (variante, nom), modeles in modeles_plis.items():
    X = original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    for famille, colonnes in FAMILLES.items():
        presentes = [c for c in colonnes if c in X.columns]
        if presentes:
            importance_familles[(variante, nom, famille)] = perte_par_permutation(modeles, X, presentes)
    for colonne in X.columns:
        importance_colonnes[(variante, nom, colonne)] = perte_par_permutation(modeles, X, [colonne])
    print(f"{variante}, {nom} : fait")

PAY_1 d'origine, LogisticRegression : fait
PAY_1 d'origine, MLPClassifier : fait
PAY_1 d'origine, RandomForest : fait
PAY_1 d'origine, CatBoost : fait
PAY_1 corrigée, LogisticRegression : fait
PAY_1 corrigée, MLPClassifier : fait
PAY_1 corrigée, RandomForest : fait
PAY_1 corrigée, CatBoost : fait


In [10]:
# Par famille : une colonne par variante et par modèle, une ligne par famille
colonnes_tableau = list(dict.fromkeys((variante, nom) for variante, nom, _ in importance_familles))
perte = pd.DataFrame({cle: {famille: np.mean(importance_familles[(*cle, famille)])
                            for famille in FAMILLES if (*cle, famille) in importance_familles} for cle in colonnes_tableau})
ecart = pd.DataFrame({cle: {famille: np.std(importance_familles[(*cle, famille)])
                            for famille in FAMILLES if (*cle, famille) in importance_familles} for cle in colonnes_tableau})
part = perte.clip(lower=0) / perte.clip(lower=0).sum() * 100

display(Markdown("**Perte de précision moyenne quand la famille est mélangée** (moyenne ± écart-type sur les 5 plis ; « — » : famille absente de la variante)"))
texte = (perte.round(3).astype(str) + " ± " + ecart.round(3).astype(str)).where(perte.notna(), "—")
display(texte.rename_axis(index="Famille", columns=["Variante", "Modèle"]))
display(Markdown("**Part de chaque famille dans la perte totale** (%, pertes négatives comptées à zéro ; à lire colonne par colonne)"))
display(part.round(1).rename_axis(index="Famille", columns=["Variante", "Modèle"]))

**Perte de précision moyenne quand la famille est mélangée** (moyenne ± écart-type sur les 5 plis ; « — » : famille absente de la variante)

Variante                                               PAY_1 d'origine                                                  PAY_1 corrigée                                             
Modèle                                              LogisticRegression  MLPClassifier   RandomForest       CatBoost LogisticRegression  MLPClassifier   RandomForest       CatBoost
Famille                                                                                                                                                                            
Codification de septembre (PAY_1 ou PAY_1_corrigee)      0.193 ± 0.007  0.176 ± 0.013  0.125 ± 0.001  0.154 ± 0.006      0.216 ± 0.012  0.202 ± 0.014   0.12 ± 0.006  0.164 ± 0.007
Codifications d'avril à août (PAY_2 à PAY_6)             0.036 ± 0.006  0.066 ± 0.008  0.126 ± 0.004  0.083 ± 0.005       0.038 ± 0.01  0.072 ± 0.011   0.11 ± 0.011  0.076 ± 0.006
Factures (BILL_AMTn)                                     0.016 ± 0.003   0.02 ± 0.004  0.009 ± 0.002  0.026 ± 0.004      0.007 ± 0.002  0.017 ± 0.002   0.01 ± 0.004  0.026 ± 0.005
Paiements (PAY_AMTn)                                     0.072 ± 0.007   0.05 ± 0.009  0.029 ± 0.002  0.048 ± 0.001      0.059 ± 0.005   0.048 ± 0.01  0.036 ± 0.004  0.049 ± 0.005
Plafond (LIMIT_BAL)                                      0.007 ± 0.002  0.011 ± 0.001  0.006 ± 0.002  0.013 ± 0.004      0.004 ± 0.003   0.01 ± 0.002  0.007 ± 0.001  0.014 ± 0.003
Démographie                                              0.006 ± 0.001  0.008 ± 0.002  0.002 ± 0.001  0.005 ± 0.001      0.006 ± 0.003  0.007 ± 0.003  0.003 ± 0.002  0.006 ± 0.002

**Part de chaque famille dans la perte totale** (%, pertes négatives comptées à zéro ; à lire colonne par colonne)

Variante                                               PAY_1 d'origine                                         PAY_1 corrigée                                    
Modèle                                              LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost
Famille                                                                                                                                                          
Codification de septembre (PAY_1 ou PAY_1_corrigee)               58.6          53.2         42.0     46.8               65.5          56.7         41.9     48.9
Codifications d'avril à août (PAY_2 à PAY_6)                      10.9          20.0         42.4     25.1               11.4          20.3         38.3     22.7
Factures (BILL_AMTn)                                               4.9           5.9          3.0      8.0                2.2           4.7          3.6      7.7
Paiements (PAY_AMTn)                                              21.8          15.1          9.8     14.5               17.9          13.5         12.8     14.6
Plafond (LIMIT_BAL)                                                2.1           3.3          2.2      4.1                1.3           2.9          2.4      4.3
Démographie                                                        1.8           2.5          0.7      1.6                1.8           2.0          1.0      1.8

In [11]:
# Par colonne, mélangée seule : perte de précision moyenne sur les 5 plis, triée sur la moyenne des deux versions et des 4 modèles
par_colonne = pd.DataFrame({(variante, nom): {colonne: np.mean(importance_colonnes[(variante, nom, colonne)])
                                              for v, n, colonne in importance_colonnes if (v, n) == (variante, nom)}
                            for variante, nom in colonnes_tableau})
ordre_colonnes = par_colonne.mean(axis=1).sort_values(ascending=False).index
with pd.option_context("display.max_rows", None):
    display(Markdown("**Perte de précision moyenne quand la colonne est mélangée seule** (moyenne sur les 5 plis ; vide : colonne absente de la variante)"))
    display(par_colonne.loc[ordre_colonnes].round(3).rename_axis(index="Colonne", columns=["Variante", "Modèle"]))

**Perte de précision moyenne quand la colonne est mélangée seule** (moyenne sur les 5 plis ; vide : colonne absente de la variante)

Variante          PAY_1 d'origine                                         PAY_1 corrigée                                    
Modèle         LogisticRegression MLPClassifier RandomForest CatBoost LogisticRegression MLPClassifier RandomForest CatBoost
Colonne                                                                                                                     
PAY_1_corrigee                NaN           NaN          NaN      NaN              0.216         0.201        0.125    0.162
PAY_1                       0.189         0.173        0.128    0.152                NaN           NaN          NaN      NaN
PAY_2                       0.003         0.006        0.016    0.013              0.015         0.010        0.016    0.012
LIMIT_BAL                   0.007         0.011        0.006    0.013              0.005         0.011        0.007    0.015
PAY_AMT1                    0.012         0.006        0.004    0.006              0.003         0.009        0.008    0.008
PAY_AMT2                    0.009         0.004        0.003    0.006              0.011         0.007        0.004    0.006
PAY_3                       0.005         0.006        0.007    0.005              0.003         0.006        0.010    0.008
PAY_AMT3                    0.007         0.005        0.003    0.005              0.008         0.010        0.003    0.005
BILL_AMT1                   0.001         0.006        0.002    0.008              0.003         0.009        0.003    0.010
PAY_5                       0.004         0.004        0.003    0.004              0.005         0.010        0.004    0.005
PAY_6                       0.005         0.004        0.003    0.003              0.007         0.006        0.004    0.005
PAY_4                       0.002         0.004        0.004    0.002              0.003         0.005        0.007    0.005
BILL_AMT3                   0.006         0.002        0.002    0.002              0.004         0.003        0.001    0.003
BILL_AMT2                   0.003         0.004        0.001    0.001              0.000         0.006        0.001    0.002
BILL_AMT4                   0.003         0.003        0.001    0.001              0.002         0.006        0.002    0.002
PAY_AMT4                    0.001         0.004        0.001    0.003              0.001         0.003        0.002    0.003
EDUCATION                   0.003         0.003        0.000    0.003              0.002         0.003        0.001    0.003
PAY_AMT5                    0.001         0.003        0.001    0.001              0.001         0.003        0.002    0.001
PAY_AMT6                   -0.000         0.004        0.001    0.002             -0.000         0.002        0.002    0.001
MARRIAGE                    0.002         0.002        0.000    0.001              0.002         0.002        0.000    0.001
AGE                         0.000         0.002        0.002    0.001              0.001         0.003        0.001    0.001
BILL_AMT6                   0.002         0.001        0.001    0.000              0.001         0.002        0.001    0.001
BILL_AMT5                   0.000         0.002        0.001    0.000              0.001         0.003        0.001    0.000
SEX                         0.001         0.001        0.000    0.000              0.001         0.001        0.001    0.000

**Comment lire** : la famille « codification de septembre » est `PAY_1` dans une version et `PAY_1_corrigee` dans l'autre. Une perte plus grande avec `PAY_1_corrigee` voudrait dire que les modèles s'appuient davantage sur la version corrigée ; une perte plus petite, que la correction a retiré une partie de ce que le 1 d'origine leur disait.

## 6. Comparaison finale : `PAY_1` d'origine face à `PAY_1` corrigée, pli par pli

Pour chaque modèle (chaque version avec ses meilleurs réglages), puis en meilleur contre meilleur : le score avec `PAY_1` d'origine, avec `PAY_1_corrigee`, et le gain pli par pli (mêmes 5 plis, mêmes clients). Mesures : précision à rappel égal (50, 60, 70, 80 %), précision moyenne, ROC AUC, rappel et précision à la règle marginale. Verdict : « gain réel » si le gain moyen dépasse l'écart-type des gains, « perte » s'il est en dessous de moins cet écart-type, sinon « dans le bruit ».

In [12]:
RAPPELS = [0.50, 0.60, 0.70, 0.80]

def verdict(gains):
    moyenne, ecart_type = np.mean(gains), np.std(gains)
    return "✅ gain réel" if moyenne > ecart_type else ("❌ perte" if moyenne < -ecart_type else "➖ dans le bruit")

def mesures_plis(variante, nom):
    p, X = probas[(variante, nom)], original[COLONNES_VARIANTE[variante]].reset_index(drop=True)
    m = {f"Précision au rappel de {r:.0%}": [] for r in RAPPELS}
    m.update({"Précision moyenne": [], "ROC AUC": [], "Rappel, règle marginale": [], "Précision, règle marginale": []})
    for modele_pli, (_, validation) in zip(modeles_plis[(variante, nom)], plis):
        y_val, p_val = y[validation], p[validation]
        d_val = modele_pli.predict(X.iloc[validation]).astype(bool)
        for r in RAPPELS:
            m[f"Précision au rappel de {r:.0%}"].append(y_val[p_val >= seuil_pour_rappel(y_val, p_val, r)].mean())
        m["Précision moyenne"].append(average_precision_score(y_val, p_val))
        m["ROC AUC"].append(roc_auc_score(y_val, p_val))
        m["Rappel, règle marginale"].append(recall_score(y_val, d_val))
        m["Précision, règle marginale"].append(precision_score(y_val, d_val, zero_division=0))
    return {k: np.array(v) for k, v in m.items()}

scores = {(v, nom): mesures_plis(v, nom) for v in VARIANTES for nom in MODELES_A_TESTER}

def meilleur(variante):
    return max(MODELES_A_TESTER, key=lambda nom: scores[(variante, nom)]["Précision moyenne"].mean())

ORIGINE, CORRIGEE = list(VARIANTES)
paires = [(nom, nom, nom) for nom in MODELES_A_TESTER] + [(f"Meilleur contre meilleur ({meilleur(ORIGINE)} → {meilleur(CORRIGEE)})", meilleur(ORIGINE), meilleur(CORRIGEE))]
lignes = []
for libelle, nom_ori, nom_cor in paires:
    a, b = scores[(ORIGINE, nom_ori)], scores[(CORRIGEE, nom_cor)]
    for mesure in a:
        gains = b[mesure] - a[mesure]
        echelle, dec = (1, 3) if mesure == "ROC AUC" else (100, 2)
        lignes.append({"Modèle": libelle, "Mesure": mesure,
                       "PAY_1 d'origine": round(a[mesure].mean() * echelle, dec), "PAY_1 corrigée": round(b[mesure].mean() * echelle, dec),
                       "Gain de la version corrigée": f"{gains.mean() * echelle:+.{dec}f} ± {gains.std() * echelle:.{dec}f}",
                       "Plis gagnés": f"{(gains > 0).sum()} sur 5", "Verdict": verdict(gains)})
table = pd.DataFrame(lignes)
display(Markdown("**Verdicts** (une colonne par modèle)"))
display(table.pivot_table(index="Mesure", columns="Modèle", values="Verdict", aggfunc="first", sort=False))
display(Markdown("**Gains de la version corrigée** (points de pourcentage ; ROC AUC en valeur)"))
display(table.pivot_table(index="Mesure", columns="Modèle", values="Gain de la version corrigée", aggfunc="first", sort=False))
with pd.option_context("display.max_rows", None):
    display(Markdown("**Détail**"))
    display(table.set_index(["Modèle", "Mesure"]))

**Verdicts** (une colonne par modèle)

Modèle,LogisticRegression,MLPClassifier,RandomForest,CatBoost,Meilleur contre meilleur (CatBoost → CatBoost)
Mesure,,,,,
Précision au rappel de 50%,➖ dans le bruit,❌ perte,➖ dans le bruit,❌ perte,❌ perte
Précision au rappel de 60%,❌ perte,➖ dans le bruit,❌ perte,➖ dans le bruit,➖ dans le bruit
Précision au rappel de 70%,❌ perte,➖ dans le bruit,➖ dans le bruit,❌ perte,❌ perte
Précision au rappel de 80%,❌ perte,➖ dans le bruit,➖ dans le bruit,➖ dans le bruit,➖ dans le bruit
Précision moyenne,❌ perte,❌ perte,❌ perte,❌ perte,❌ perte
ROC AUC,❌ perte,➖ dans le bruit,❌ perte,❌ perte,❌ perte
"Rappel, règle marginale",✅ gain réel,➖ dans le bruit,➖ dans le bruit,➖ dans le bruit,➖ dans le bruit
"Précision, règle marginale",❌ perte,➖ dans le bruit,❌ perte,➖ dans le bruit,➖ dans le bruit


**Gains de la version corrigée** (points de pourcentage ; ROC AUC en valeur)

Modèle,LogisticRegression,MLPClassifier,RandomForest,CatBoost,Meilleur contre meilleur (CatBoost → CatBoost)
Mesure,,,,,
Précision au rappel de 50%,-0.15 ± 0.45,-1.00 ± 0.61,+0.00 ± 0.33,-0.83 ± 0.53,-0.83 ± 0.53
Précision au rappel de 60%,-0.51 ± 0.31,-0.10 ± 0.30,-0.45 ± 0.23,-0.22 ± 0.35,-0.22 ± 0.35
Précision au rappel de 70%,-0.51 ± 0.36,-0.18 ± 0.59,+0.28 ± 0.33,-0.56 ± 0.53,-0.56 ± 0.53
Précision au rappel de 80%,-0.30 ± 0.29,+0.59 ± 0.84,+0.21 ± 0.29,+0.06 ± 0.27,+0.06 ± 0.27
Précision moyenne,-2.24 ± 0.28,-1.65 ± 0.64,-1.94 ± 0.35,-1.36 ± 0.53,-1.36 ± 0.53
ROC AUC,-0.005 ± 0.001,-0.000 ± 0.004,-0.002 ± 0.001,-0.003 ± 0.000,-0.003 ± 0.000
"Rappel, règle marginale",+3.39 ± 2.66,-1.84 ± 4.61,+3.23 ± 4.24,-0.89 ± 2.31,-0.89 ± 2.31
"Précision, règle marginale",-4.55 ± 1.53,-1.74 ± 1.93,-5.08 ± 2.12,-2.08 ± 2.12,-2.08 ± 2.12


**Détail**

PAY_1 d'origine  PAY_1 corrigée Gain de la version corrigée Plis gagnés          Verdict
Modèle                                         Mesure                                                                                                              
LogisticRegression                             Précision au rappel de 50%           56.880          56.730                -0.15 ± 0.45     3 sur 5  ➖ dans le bruit
                                               Précision au rappel de 60%           48.930          48.410                -0.51 ± 0.31     0 sur 5          ❌ perte
                                               Précision au rappel de 70%           40.840          40.330                -0.51 ± 0.36     1 sur 5          ❌ perte
                                               Précision au rappel de 80%           34.030          33.730                -0.30 ± 0.29     1 sur 5          ❌ perte
                                               Précision moyenne                    54.620          52.370                -2.24 ± 0.28     0 sur 5          ❌ perte
                                               ROC AUC                               0.775           0.770              -0.005 ± 0.001     0 sur 5          ❌ perte
                                               Rappel, règle marginale              38.950          42.340                +3.39 ± 2.66     4 sur 5      ✅ gain réel
                                               Précision, règle marginale           65.690          61.130                -4.55 ± 1.53     0 sur 5          ❌ perte
MLPClassifier                                  Précision au rappel de 50%           57.740          56.730                -1.00 ± 0.61     0 sur 5          ❌ perte
                                               Précision au rappel de 60%           49.200          49.100                -0.10 ± 0.30     2 sur 5  ➖ dans le bruit
                                               Précision au rappel de 70%           41.220          41.030                -0.18 ± 0.59     1 sur 5  ➖ dans le bruit
                                               Précision au rappel de 80%           34.170          34.760                +0.59 ± 0.84     3 sur 5  ➖ dans le bruit
                                               Précision moyenne                    55.180          53.530                -1.65 ± 0.64     0 sur 5          ❌ perte
                                               ROC AUC                               0.777           0.777              -0.000 ± 0.004     1 sur 5  ➖ dans le bruit
                                               Rappel, règle marginale              37.430          35.590                -1.84 ± 4.61     1 sur 5  ➖ dans le bruit
                                               Précision, règle marginale           65.940          64.200                -1.74 ± 1.93     2 sur 5  ➖ dans le bruit
RandomForest                                   Précision au rappel de 50%           57.480          57.490                +0.00 ± 0.33     1 sur 5  ➖ dans le bruit
                                               Précision au rappel de 60%           49.830          49.390                -0.45 ± 0.23     0 sur 5          ❌ perte
                                               Précision au rappel de 70%           41.240          41.510                +0.28 ± 0.33     4 sur 5  ➖ dans le bruit
                                               Précision au rappel de 80%           33.950          34.150                +0.21 ± 0.29     4 sur 5  ➖ dans le bruit
                                               Précision moyenne                    54.800          52.860                -1.94 ± 0.35     0 sur 5          ❌ perte
                                               ROC AUC                               0.779           0.777              -0.002 ± 0.001     0 sur 5          ❌ perte
                                               Rappel, règle marginale              38.650          41.880                +3.23 ± 4.24     

**Comment lire** :
- **Précision au rappel de 50 à 80 %** : les deux versions détectent la même part des défauts ; un gain positif veut dire que, pour en détecter autant, la version corrigée signale moins de bons clients.
- **Précision moyenne, ROC AUC** : la capacité à classer les défauts en tête, tous seuils confondus.
- **Règle marginale** : rappel et précision au seuil choisi par la règle ; ils se lisent ensemble.
- **Chaque version a ses propres réglages** (boucle lancée séparément) : on compare chaque modèle au mieux de ce qu'il fait avec chaque colonne.

## 7. Conclusion

*À écrire par l'utilisateur après lecture des résultats.*